In [5]:
import numpy as np
import pandas as pd
import requests
import getpass
from typing import List
import json

In [6]:
df = pd.read_csv('./datasets/AAPL.csv')
print("Shape: ", df.shape)

df['Date'] = df['Date'].str[:10]
print(df['Date'].min(), df['Date'].max())
df.head()

Shape:  (11082, 8)
1980-12-12 2024-11-26


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1980-12-12,0.098834,0.099264,0.098834,0.098834,469033600,0.0,0.0
1,1980-12-15,0.094108,0.094108,0.093678,0.093678,175884800,0.0,0.0
2,1980-12-16,0.087232,0.087232,0.086802,0.086802,105728000,0.0,0.0
3,1980-12-17,0.088951,0.089381,0.088951,0.088951,86441600,0.0,0.0
4,1980-12-18,0.091530,0.091959,0.091530,0.091530,73449600,0.0,0.0


In [7]:
df = df[df['Date'] > '2004-11-26']

### Remove rows earlier than 2004-11-26

In [8]:
print(f"Shape: {df.shape}")
df['Date'].min(), df['Date'].max()

Shape: (5034, 8)


('2004-11-29', '2024-11-26')

In [34]:
api_key = getpass.getpass('Enter your Alpha Vantage API key: ')
ALPHAVANTAGE_API_URL = f"https://www.alphavantage.co/query?function=FEDERAL_FUNDS_RATE&interval=monthly&apikey=${api_key}"

def get_interest_rates():
    response = requests.get(ALPHAVANTAGE_API_URL)
    data = response.json()
    rates = data['data']
    return [{"date": rate['date'], "value": float(rate['value'])} for rate in rates]

rates = get_interest_rates()
with open('datasets/interest_rates.json', mode='w') as fp:
    json.dump(rates, fp)

In [9]:
# Load interest rates from file

rates = pd.read_json('datasets/interest_rates.json')
rates

,date,value
0,2024-11-01,4.64
1,2024-10-01,4.83
2,2024-09-01,5.13
3,2024-08-01,5.33
4,2024-07-01,5.33
...,...,...
840,1954-11-01,0.83
841,1954-10-01,0.85
842,1954-09-01,1.07
843,1954-08-01,1.22


### Merge interest rate and stock price data

In [11]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

rates['date'] = pd.to_datetime(rates['date'])
rates['Year'] = rates['date'].dt.year
rates['Month'] = rates['date'].dt.month

df_merged = pd.merge(df, rates, on=['Year', 'Month'], how='inner')

df_merged.drop(columns=['date', 'Year', 'Month'], inplace=True)
df_merged.rename(columns={'value': 'Interest Rate'}, inplace=True)

# Convert to percentage 0..1 instead of 0..100
df_merged['Interest Rate'] = df_merged['Interest Rate'] / 100

df_merged.head()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Interest Rate
0,2004-11-29,1.038618,1.047957,1.015420,1.030936,1712916800,0.0,0.0,0.0193
1,2004-11-30,1.036208,1.036208,1.009997,1.009997,1028518400,0.0,0.0,0.0193
2,2004-12-01,1.021144,1.023554,0.998248,1.021144,800553600,0.0,0.0,0.0216
3,2004-12-02,0.996139,1.007738,0.973996,0.982280,987442400,0.0,0.0,0.0216
4,2004-12-03,0.972037,0.979117,0.930162,0.944170,1238848800,0.0,0.0,0.0216
